# 🧱 **Module 4: Advanced Debugging — The Factory Outage**
## Capstone Incident Queue

**Duration:** 45 minutes | **Level:** 300–400

---

### Scenario

It's Monday morning and the LEGO data platform is on fire. Multiple pipelines are failing
or running 10× slower than normal. You are the on-call engineer.

Your job: **triage, diagnose, fix, and validate** each incident using the skills from Modules 1–3.

### The Triage Loop

Every incident follows the same methodology:

| Step | What you do |
|------|------------|
| 📋 **Read** | Understand the symptom and SLA impact |
| 🔍 **Inspect** | Examine Spark UI, physical plan, table metadata |
| 🧠 **Identify** | Categorize the bottleneck (I/O, memory, shuffle, scheduling, storage) |
| 🔧 **Fix** | Apply the remediation |
| ✅ **Validate** | Measure before/after improvement |

### Incident Queue

| # | Severity | Incident | Pipeline |
|---|----------|----------|----------|
| 1 | 🔴 Critical | OOM Crash | Daily Quality Report |
| 2 | 🟠 High | 10× Slowdown | Customer 360 |
| 3 | 🟡 Medium | Mysterious Spill | Inventory Reconciliation |
| 4 | 🟡 Optional | Partition Explosion | Parts Demand Forecast |
| 5 | 🟢 Optional | Delta Storage Regression | Manufacturing Analytics |

**Incidents 1–3 are mandatory.** Incidents 4–5 are stretch goals for fast finishers.

---

In [ ]:
%run _benchmark_utils

In [ ]:
# ============================================================
# SETUP — Helpers and Benchmark Proxy
# ============================================================
import time
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.functions import spark_partition_id
from delta.tables import DeltaTable

SCHEMA = "bronze"             # source tables from unoptimized SJD
CAPSTONE_SCHEMA = "capstone"  # scenario-specific clones for incidents

if "_BenchmarkProxy" not in globals():
    raise NotImplementedError("_benchmark_utils was not run! Run the prior cell!")

print("\u2705 Setup complete \u2014 helpers loaded")
print(f"   Source schema: {SCHEMA}")
print(f"   Capstone schema: {CAPSTONE_SCHEMA}")

---

## 🏗️ Scenario Setup

Before triaging incidents, we need to prepare the broken pipeline scenarios.
Each incident uses shallow-cloned tables with intentional problems baked in.

> **In a real workshop**, these tables are pre-staged by the instructor.
> Here we create them explicitly so you can see the setup.

In [ ]:
# ============================================================
# SCENARIO SETUP — Create broken tables for each incident
# ============================================================

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CAPSTONE_SCHEMA}")

# --- INC-1: OOM Setup ---
spark.sql(f"DROP TABLE IF EXISTS {CAPSTONE_SCHEMA}.production_order_large")
spark.sql(f"""
    CREATE TABLE {CAPSTONE_SCHEMA}.production_order_large
    SHALLOW CLONE {SCHEMA}.production_order
""")
spark.sql(f"DROP TABLE IF EXISTS {CAPSTONE_SCHEMA}.quality_inspection")
spark.sql(f"""
    CREATE TABLE {CAPSTONE_SCHEMA}.quality_inspection
    SHALLOW CLONE {SCHEMA}.quality_inspection
""")

po_count = spark.table(f"{CAPSTONE_SCHEMA}.production_order_large").count()
qi_count = spark.table(f"{CAPSTONE_SCHEMA}.quality_inspection").count()
print(f"  \u2705 INC-1: production_order_large ({po_count:,} rows), quality_inspection ({qi_count:,} rows)")

# --- INC-2: Slowdown Setup ---
for t in ["customer", "web_order", "web_order_line", "product_return"]:
    spark.sql(f"DROP TABLE IF EXISTS {CAPSTONE_SCHEMA}.{t}")
    spark.sql(f"""
        CREATE TABLE {CAPSTONE_SCHEMA}.{t}
        SHALLOW CLONE {SCHEMA}.{t}
    """)

wo_metrics = get_table_metrics(f"{CAPSTONE_SCHEMA}.web_order")
print(f"  \u2705 INC-2: web_order has {wo_metrics['num_files']:,} files (avg {wo_metrics['avg_file_kb']:.0f} KB) \u2014 small file problem")

# --- INC-3: Spill Setup ---
spark.sql(f"DROP TABLE IF EXISTS {CAPSTONE_SCHEMA}.inventory_transaction")
spark.sql(f"""
    CREATE TABLE {CAPSTONE_SCHEMA}.inventory_transaction
    SHALLOW CLONE {SCHEMA}.inventory_transaction
""")

inv_count = spark.table(f"{CAPSTONE_SCHEMA}.inventory_transaction").count()
print(f"  \u2705 INC-3: inventory_transaction ({inv_count:,} rows)")

# --- INC-4 (optional): uses same inventory_transaction
print(f"  \u2705 INC-4: (uses inventory_transaction from INC-3)")

# --- INC-5 (optional): Delta Storage Regression ---
spark.sql(f"DROP TABLE IF EXISTS {CAPSTONE_SCHEMA}.manufacturing_event")
spark.sql(f"""
    CREATE TABLE {CAPSTONE_SCHEMA}.manufacturing_event
    SHALLOW CLONE {SCHEMA}.manufacturing_event
""")
spark.sql(f"""
    ALTER TABLE {CAPSTONE_SCHEMA}.manufacturing_event
    SET TBLPROPERTIES (
        'delta.autoOptimize.autoCompact' = 'false',
        'delta.autoOptimize.optimizeWrite' = 'false'
    )
""")

me_metrics = get_table_metrics(f"{CAPSTONE_SCHEMA}.manufacturing_event")
print(f"  \u2705 INC-5: manufacturing_event ({me_metrics['num_files']:,} files, optimizations disabled)")
print(f"\n\U0001f4cc All incidents use schema: {CAPSTONE_SCHEMA}")

### 🔍 The Data You're Debugging

Before triaging incidents, let's see what's in these tables. This is **real LEGO data** — actual sets, manufacturing events, and customer orders.

In [ ]:
# What's being manufactured?
print("🏭 Recent Manufacturing Events (with defects highlighted)\n")
display(spark.sql(f"""
    SELECT me.machine_id, me.part_num, p.name AS part_name,
           me.defect_detected, me.defect_type,
           ROUND(me.mold_temp, 1) AS mold_temp,
           ROUND(me.cycle_time_ms / 1000.0, 1) AS cycle_time_sec
    FROM {CAPSTONE_SCHEMA}.manufacturing_event me
    LEFT JOIN {SCHEMA}.parts p ON me.part_num = p.part_num
    WHERE me.defect_detected = true
    ORDER BY me.timestamp DESC
    LIMIT 10
"""))

# What are customers buying?
print("\n🛒 Recent Orders — Real LEGO Sets\n")
display(spark.sql(f"""
    SELECT c.name AS customer, c.loyalty_tier,
           wol.item_name, wol.quantity, ROUND(wol.extended_price, 2) AS price,
           wo.shipping_country
    FROM {CAPSTONE_SCHEMA}.web_order_line wol
    JOIN {CAPSTONE_SCHEMA}.web_order wo ON wol.order_id = wo.order_id
    JOIN {CAPSTONE_SCHEMA}.customer c ON wo.customer_id = c.customer_id
    ORDER BY wol.extended_price DESC
    LIMIT 10
"""))

---

## 🔴 INC-1: OOM Crash — "Daily Quality Report"

### 📋 On-Call Ticket

| Field | Details |
|-------|---------|
| **Severity** | 🔴 Critical — pipeline failing |
| **SLA Impact** | Daily quality report not delivered; production decisions blocked |
| **Symptom** | Executor OOM on the "Daily Quality Report" job that joins `quality_inspection` with `production_order` |
| **Suspected Pipeline** | `quality_inspection ⟕ production_order` join |
| **Context** | A recent code change introduced an explicit `broadcast()` hint on `production_order`. The table has grown significantly since the hint was added. |

### 🔍 Your Task

1. **Read** the broken pipeline code below
2. **Examine** the physical plan — what join strategy does Spark use?
3. **Check** the table sizes — is the broadcast appropriate?
4. **Diagnose** why the OOM occurs
5. **Fix** the code and validate

> ⚠️ **Safety note:** We won't trigger a real OOM on shared compute. Instead, examine the
> plan and table metadata to diagnose the issue, then run a safe validation.

In [ ]:
# ============================================================
# INC-1: THE BROKEN PIPELINE (examine, don't run at scale!)
# ============================================================
# Someone added broadcast() on production_order to "speed up" the join.
# That worked when the table had 1,000 rows. Now it has grown.

from pyspark.sql.functions import broadcast

po = spark.table(f"{CAPSTONE_SCHEMA}.production_order_large")
qi = spark.table(f"{CAPSTONE_SCHEMA}.quality_inspection")

# 💀 THE PROBLEMATIC CODE — explicit broadcast on a large table
quality_report = qi.join(
    broadcast(po),  # <-- Forces the ENTIRE production_order table to be broadcast
    qi.production_order_id == po.production_order_id,
    "inner"
).select(
    qi.inspection_id,
    qi.timestamp,
    po.part_num,
    po.machine_id,
    qi.result,
    qi.clutch_power_score,
    qi.dimensional_accuracy,
)

# Step 1: Check how big the broadcast table actually is
print("\U0001f4ca Table sizes:")
show_metrics(f"{CAPSTONE_SCHEMA}.production_order_large", "broadcast target")
show_metrics(f"{CAPSTONE_SCHEMA}.quality_inspection")

# Step 2: Look at the physical plan — spot the BroadcastHashJoin
print("\n\U0001f4cb Physical Plan (look for BroadcastHashJoin):")
quality_report.explain()

### 🧠 Diagnosis Questions

Before applying the fix, answer these:

1. What join strategy does Spark show in the plan? (`BroadcastHashJoin` or `SortMergeJoin`?)
2. How large is `production_order_large`? Is it small enough to broadcast safely?
3. What is the default `spark.sql.autoBroadcastJoinThreshold`? (10 MB)
4. Why would increasing `autoBroadcastJoinThreshold` make this WORSE, not better?

<details>
<summary>💡 Hint (click to reveal)</summary>

The explicit `broadcast()` hint overrides all thresholds — it forces Spark to broadcast
the table regardless of size. The fix is to **remove the hint**, not change the threshold.
With AQE enabled, Spark will choose SortMergeJoin for large tables automatically.

</details>

### 🎯 Challenge: Fix the OOM

You've identified the problem: `broadcast(po)` forces the entire `production_order_large` table into driver memory. The table is too big for that.

**Your task:** Rewrite the join below WITHOUT the broadcast hint. Let Spark/AQE choose the join strategy automatically.

> 💡 Hint: Just remove `broadcast()` — the rest of the join stays the same.

In [ ]:
# YOUR CODE HERE
# Rewrite the join from INC-1 without the broadcast hint
# quality_report_fixed = qi.join(
#     ???,
#     qi.production_order_id == po.production_order_id,
#     "inner"
# ).select(...)

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
quality_report_fixed = qi.join(
    po,  # No broadcast() wrapper!
    qi.production_order_id == po.production_order_id,
    "inner"
).select(
    qi.inspection_id,
    qi.timestamp,
    po.part_num,
    po.machine_id,
    qi.result,
    qi.clutch_power_score,
    qi.dimensional_accuracy,
)
```

**When to use broadcast:**
- Table is small enough to fit in executor memory (< ~100 MB)
- You've verified the size won't grow
- Never broadcast a table that could grow unbounded!

</details>

---

In [ ]:
# ============================================================
# INC-1: THE FIX — Remove the broadcast hint
# ============================================================

# ✅ Fixed: Let Spark/AQE choose the join strategy based on actual table sizes
quality_report_fixed = qi.join(
    po,  # No broadcast hint — AQE will choose SortMergeJoin for large tables
    qi.production_order_id == po.production_order_id,
    "inner"
).select(
    qi.inspection_id,
    qi.timestamp,
    po.part_num,
    po.machine_id,
    qi.result,
    qi.clutch_power_score,
    qi.dimensional_accuracy,
)

# Verify: plan should now show SortMergeJoin (not BroadcastHashJoin)
print("\U0001f4cb Fixed Physical Plan (should show SortMergeJoin):")
quality_report_fixed.explain()

# Safe validation — run the fixed query
print()
#quality_report_fixed.benchmark("INC-1: OOM Fix", "fixed (no broadcast)").count()

In [ ]:
# 🎯 Let's see the quality report we just fixed!
print("📋 Quality Report — Inspection Results by Machine\n")
display(quality_report_fixed.select(
    "machine_id", "part_num", "result",
    F.round("clutch_power_score", 2).alias("clutch_power"),
    F.round("dimensional_accuracy", 4).alias("dim_accuracy")
).orderBy(F.desc("clutch_power")).limit(15))

---

## 🟠 INC-2: 10× Slowdown — "Customer 360 Pipeline"

### 📋 On-Call Ticket

| Field | Details |
|-------|---------|
| **Severity** | 🟠 High — pipeline running 10× slower than SLA |
| **SLA Impact** | Customer 360 view not ready for marketing team by 9 AM |
| **Symptom** | Multi-table join (`customer → web_order → web_order_line → product_return`) takes 45 min instead of 5 min |
| **Suspected Pipeline** | Customer 360 denormalized view |
| **Context** | No code changes recently. The unoptimized SJD has been writing data for days without maintenance. |

### 🔍 Your Task

1. **Run** the broken pipeline and capture baseline timing
2. **Investigate** — there are **TWO compounding issues**, find them both:
   - Check file counts and sizes with `DESCRIBE DETAIL`
   - Check table history with `DESCRIBE HISTORY`
   - Check join strategies with `explain()`
   - Count files being scanned with `inputFiles()`
3. **Fix** both issues
4. **Validate** the improvement

In [ ]:
# ============================================================
# INC-2: THE BROKEN PIPELINE — run and capture baseline
# ============================================================

customer_360 = (
    spark.table(f"{CAPSTONE_SCHEMA}.customer").alias("c")
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.web_order").alias("wo"),
        F.col("c.customer_id") == F.col("wo.customer_id"),
        "inner"
    )
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.web_order_line").alias("wol"),
        F.col("wo.order_id") == F.col("wol.order_id"),
        "inner"
    )
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.product_return").alias("pr"),
        F.col("wo.order_id") == F.col("pr.order_id"),
        "left"
    )
    .select(
        "c.customer_id", "c.name", "c.country", "c.loyalty_tier",
        "wo.order_id", "wo.order_date", "wo.order_total",
        "wol.set_num", "wol.item_name", "wol.quantity", "wol.extended_price",
        "pr.return_id", "pr.reason", "pr.refund_amount",
    )
)

# Baseline timing
customer_360.benchmark("INC-2: Customer 360", "before (broken)").count()

### 🎯 Challenge: Find BOTH Root Causes

The Customer 360 pipeline is running 10× slower than normal. There are **two** independent root causes.

**Your task:** Write diagnostic queries to investigate:
1. Use `DESCRIBE DETAIL` to check file sizes for each table
2. Check `DESCRIBE HISTORY` to see when tables were last optimized
3. Use `.explain()` on the `customer_360` DataFrame to see the join strategy

> 💡 Hint: Look for both a **storage** problem and a **statistics** problem.

In [ ]:
# YOUR CODE HERE
# Investigate the tables used in the Customer 360 pipeline
# Tables: customer, web_order, web_order_line, product_return

# Check file metrics for each table
# for t in ["customer", "web_order", "web_order_line", "product_return"]:
#     display(spark.sql(f"DESCRIBE DETAIL {CAPSTONE_SCHEMA}.{t}").select("name", "numFiles", "sizeInBytes"))

# Check the physical plan
# customer_360.explain()

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal both root causes</summary>

<br/>

**Root Cause 1: Small Files**
```python
for t in ["customer", "web_order", "web_order_line", "product_return"]:
    show_metrics(CAPSTONE_SCHEMA, t)
```
→ `web_order` has thousands of tiny files (avg < 10 KB). Each file requires a separate I/O operation.

**Root Cause 2: Stale Statistics**
```python
customer_360.explain()
```
→ The plan shows `SortMergeJoin` for `product_return`, which is a small table. Spark should use `BroadcastHashJoin`, but it can't because the table statistics are outdated.

**Fix requires BOTH:**
1. `OPTIMIZE` to compact small files
2. `ANALYZE TABLE ... COMPUTE STATISTICS` to refresh stats

</details>

---

In [ ]:
# ============================================================
# INC-2: DIAGNOSIS — Find BOTH root causes
# ============================================================

print("=" * 60)
print("EVIDENCE 1: File counts and sizes")
print("=" * 60)
for t in ["customer", "web_order", "web_order_line", "product_return"]:
    show_metrics(CAPSTONE_SCHEMA, t)

print()
print("=" * 60)
print("EVIDENCE 2: How many files is the query scanning?")
print("=" * 60)
for t in ["web_order", "product_return"]:
    file_count = spark.table(f"{CAPSTONE_SCHEMA}.{t}").selectExpr("input_file_name()").distinct().count()
    print(f"   {t}: scanning {file_count:,} files")

print()
print("=" * 60)
print("EVIDENCE 3: Physical plan — what join strategies?")
print("=" * 60)
customer_360.explain()

print()
print("=" * 60)
print("EVIDENCE 4: Table history — when was last OPTIMIZE?")
print("=" * 60)
spark.sql(f"DESCRIBE HISTORY {CAPSTONE_SCHEMA}.web_order").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(10, truncate=False)

### 🧠 Diagnosis Questions

1. How many files does `web_order` have? What's the average file size?
2. Is `product_return` small enough for a broadcast join? What join did Spark choose?
3. Has `web_order` ever been `OPTIMIZE`d? (Check `DESCRIBE HISTORY`)
4. What are the TWO root causes?

<details>
<summary>💡 Hint (click to reveal)</summary>

**Root Cause 1:** `web_order` has thousands of tiny files from the unoptimized pipeline.
Every join scan must open each file individually — massive I/O overhead.

**Root Cause 2:** Statistics are stale (or never collected). `product_return` is small enough
for a broadcast join, but Spark doesn't know that and chooses a SortMergeJoin instead.
Fix: `ANALYZE TABLE` to refresh statistics.

</details>

### 🎯 Challenge: Fix the Customer 360 Pipeline

You found two root causes. **Your task:** Fix both:
1. Run `OPTIMIZE` on `web_order` to compact small files
2. Run `ANALYZE TABLE` on `product_return` to refresh statistics

> 💡 Hint: `ANALYZE TABLE {schema}.{table} COMPUTE STATISTICS` updates the stats catalog.

In [ ]:
# YOUR CODE HERE
# Fix 1: Compact small files in web_order
# spark.sql(f"OPTIMIZE ...")

# Fix 2: Refresh statistics on product_return
# spark.sql(f"ANALYZE TABLE ...")

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
# Fix 1: Compact small files
spark.sql(f"OPTIMIZE {CAPSTONE_SCHEMA}.web_order")

# Fix 2: Refresh statistics so Spark knows product_return is small
spark.sql(f"ANALYZE TABLE {CAPSTONE_SCHEMA}.product_return COMPUTE STATISTICS")
```

After these fixes:
- `web_order` reads will be faster (fewer, larger files)
- Spark will auto-broadcast `product_return` (BroadcastHashJoin → much faster join)

</details>

---

In [ ]:
# ============================================================
# INC-2: THE FIX — Compact small files AND refresh statistics
# ============================================================

# Fix 1: Compact the small files
print("\U0001f527 Fix 1: OPTIMIZE web_order (compact small files)...")
spark.sql(f"OPTIMIZE {CAPSTONE_SCHEMA}.web_order")
show_metrics(CAPSTONE_SCHEMA, "web_order", "after OPTIMIZE")

print()

# Fix 2: Refresh statistics so optimizer knows product_return is small
print("\U0001f527 Fix 2: ANALYZE TABLE product_return (refresh statistics)...")
spark.sql(f"ANALYZE TABLE {CAPSTONE_SCHEMA}.product_return COMPUTE STATISTICS FOR ALL COLUMNS")
print("   Statistics refreshed.")

In [ ]:
# ============================================================
# INC-2: VALIDATE — Re-run and compare
# ============================================================

customer_360_fixed = (
    spark.table(f"{CAPSTONE_SCHEMA}.customer").alias("c")
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.web_order").alias("wo"),
        F.col("c.customer_id") == F.col("wo.customer_id"),
        "inner"
    )
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.web_order_line").alias("wol"),
        F.col("wo.order_id") == F.col("wol.order_id"),
        "inner"
    )
    .join(
        spark.table(f"{CAPSTONE_SCHEMA}.product_return").alias("pr"),
        F.col("wo.order_id") == F.col("pr.order_id"),
        "left"
    )
    .select(
        "c.customer_id", "c.name", "c.country", "c.loyalty_tier",
        "wo.order_id", "wo.order_date", "wo.order_total",
        "wol.set_num", "wol.item_name", "wol.quantity", "wol.extended_price",
        "pr.return_id", "pr.reason", "pr.refund_amount",
    )
)

# Verify: plan should now show BroadcastHashJoin for product_return
print("\U0001f4cb Fixed Physical Plan (look for BroadcastHashJoin on product_return):")
customer_360_fixed.explain()
print()

# Compare timing
customer_360_fixed.benchmark("INC-2: Customer 360", "after (OPTIMIZE + ANALYZE)").count()

In [ ]:
# 🛒 The Customer 360 pipeline produces real insights!
print("👑 Top Customers by Revenue\n")
display(spark.sql(f"""
    SELECT c.name, c.loyalty_tier, c.country,
           COUNT(DISTINCT wo.order_id) AS orders,
           ROUND(SUM(wol.extended_price), 2) AS total_spent
    FROM {CAPSTONE_SCHEMA}.customer c
    JOIN {CAPSTONE_SCHEMA}.web_order wo ON c.customer_id = wo.customer_id
    JOIN {CAPSTONE_SCHEMA}.web_order_line wol ON wo.order_id = wol.order_id
    GROUP BY c.name, c.loyalty_tier, c.country
    ORDER BY total_spent DESC
    LIMIT 10
"""))

print("\n📦 Most-Returned Items\n")
display(spark.sql(f"""
    SELECT pr.reason, COUNT(*) AS return_count,
           ROUND(SUM(pr.refund_amount), 2) AS total_refunded
    FROM {CAPSTONE_SCHEMA}.product_return pr
    GROUP BY pr.reason
    ORDER BY return_count DESC
"""))

---

## 🟡 INC-3: Mysterious Spill — "Inventory Reconciliation"

### 📋 On-Call Ticket

| Field | Details |
|-------|---------|
| **Severity** | 🟡 Medium — pipeline completing but very slowly |
| **SLA Impact** | Inventory reconciliation 4× slower than expected |
| **Symptom** | Window function calculating running balances spills heavily to disk |
| **Suspected Pipeline** | Running balance calculation over `inventory_transaction` |
| **Context** | A few very popular LEGO parts dominate the inventory volume. The window function must sort all transactions per part — but some partitions are massive. |

### 🔍 Your Task

1. **Run** the broken pipeline and capture baseline timing
2. **Check** Spark UI for spill metrics and task duration variance
3. **Inspect** partition sizes to find the skew
4. **Fix** the skew and validate

In [ ]:
# ============================================================
# INC-3: THE BROKEN PIPELINE — skewed window function
# ============================================================

from pyspark.sql.window import Window

inv = spark.table(f"{CAPSTONE_SCHEMA}.inventory_transaction")

# Window function: running balance per (part_num, color_id) ordered by timestamp
# Problem: a few hot parts have millions of transactions -> huge partitions
running_balance_window = Window.partitionBy("part_num", "color_id").orderBy("timestamp")

inventory_reconciliation = (
    inv
    .withColumn("RunningBalance", F.sum("quantity").over(running_balance_window))
    .withColumn("TxnCount", F.count("*").over(running_balance_window))
)

# Baseline timing
inventory_reconciliation.benchmark("INC-3: Inventory Recon", "before (skewed)").count()

### 🎯 Challenge: Find the Hot Keys

The inventory reconciliation is spilling to disk. You suspect data skew in the window function's partition key.

**Your task:** Write a query to find which `part_num` values have the most inventory transactions. Show the top 20.

> 💡 Hint: `GROUP BY part_num` with `COUNT(*)`, ordered descending.

In [ ]:
# YOUR CODE HERE
# Find the top 20 most-transacted parts
# display(inv.groupBy(...).count().orderBy(...))

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
display(inv.groupBy("part_num").count().orderBy(F.desc("count")).limit(20))
```

You'll see a classic **Zipf distribution** — a few parts have orders of magnitude more transactions than the rest. These "hot keys" create oversized partitions in the window function, causing spill.

</details>

---

In [ ]:
# ============================================================
# INC-3: DIAGNOSIS — Find the skew
# ============================================================

print("=" * 60)
print("EVIDENCE 1: Transaction count per part_num (top 20)")
print("=" * 60)
inv.groupBy("part_num").count().orderBy(F.desc("count")).show(20)

print()
print("=" * 60)
print("EVIDENCE 2: Partition size distribution")
print("=" * 60)
# Repartition by the window key and check sizes
inv_repartitioned = inv.repartition("part_num", "color_id")
partition_sizes = (
    inv_repartitioned
    .withColumn("_pid", spark_partition_id())
    .groupBy("_pid")
    .count()
    .orderBy(F.desc("count"))
)
print(f"   Total partitions: {inv_repartitioned.rdd.getNumPartitions()}")
partition_sizes.show(10)

print()
print("   Stats on partition sizes:")
partition_sizes.select(
    F.min("count").alias("min_rows"),
    F.max("count").alias("max_rows"),
    F.mean("count").alias("avg_rows"),
    F.stddev("count").alias("stddev_rows"),
).show()

In [ ]:
# 🧱 What are these hot parts? Let's look up their names!
print("🔥 Hottest Parts by Transaction Volume\n")
display(spark.sql(f"""
    SELECT it.part_num, p.name AS part_name, p.part_cat_id,
           COUNT(*) AS txn_count,
           SUM(CASE WHEN it.quantity > 0 THEN it.quantity ELSE 0 END) AS total_in,
           SUM(CASE WHEN it.quantity < 0 THEN ABS(it.quantity) ELSE 0 END) AS total_out
    FROM {CAPSTONE_SCHEMA}.inventory_transaction it
    LEFT JOIN {SCHEMA}.parts p ON it.part_num = p.part_num
    GROUP BY it.part_num, p.name, p.part_cat_id
    ORDER BY txn_count DESC
    LIMIT 10
"""))
print("💡 These common bricks (2x4, 2x2, plates) dominate inventory movement!")

### 🧠 Diagnosis Questions

1. Which parts have the most transactions? How much bigger are they than average?
2. What is the ratio of largest to smallest partition? (This is the skew factor)
3. Why does skew cause spill? (The largest partition must fit in a single task's memory)
4. Would simply adding more executors help? (No — skew is per-partition, not per-cluster)

<details>
<summary>💡 Hint (click to reveal)</summary>

**The fix:** Salt the hot keys to distribute their rows across multiple partitions,
then remove the salt after the window calculation. This is the classic "salting" pattern
for data skew.

Alternative: Use an incremental aggregation pattern — pre-aggregate into hourly or daily
buckets first, then compute the running balance on the smaller aggregated dataset.

</details>

In [ ]:
# ============================================================
# INC-3: THE FIX — Salt hot keys to distribute skew
# ============================================================

NUM_SALT_BUCKETS = 10

# Step 1: Add a salt column to spread hot keys across buckets
inv_salted = inv.withColumn(
    "_salt", (F.abs(F.hash("transaction_id")) % NUM_SALT_BUCKETS).cast("int")
)

# Step 2: Compute running balance within salted partitions
# For SUM, we can compute partial sums per salt bucket
salted_window = Window.partitionBy("part_num", "color_id", "_salt").orderBy("timestamp")

inventory_recon_fixed = (
    inv_salted
    .withColumn("RunningBalance", F.sum("quantity").over(salted_window))
    .withColumn("TxnCount", F.count("*").over(salted_window))
    .drop("_salt")
)

# Validate: compare timing
inventory_recon_fixed.benchmark("INC-3: Inventory Recon", "after (salted)").count()

---

## 🟡 INC-4: Partition Explosion — "Parts Demand Forecast" *(Optional)*

### 📋 On-Call Ticket

| Field | Details |
|-------|---------|
| **Severity** | 🟡 Medium |
| **SLA Impact** | Forecast pipeline timing out |
| **Symptom** | Scheduler overwhelmed — tasks take 2 sec to schedule but only 0.1 sec to execute |
| **Suspected Pipeline** | `repartition()` on high-cardinality composite key |
| **Context** | Developer used `repartition("PartNum", "ColorId")` to "improve parallelism" — but there are 62K+ parts × 200+ colors. |

### 🔍 Your Task

1. Run the broken pipeline
2. Check `getNumPartitions()` — how many partitions were created?
3. Compare scheduling overhead vs. actual execution in Spark UI
4. Fix and validate

In [ ]:
# ============================================================
# INC-4: THE BROKEN PIPELINE — partition explosion
# ============================================================

inv = spark.table(f"{CAPSTONE_SCHEMA}.inventory_transaction")

# 💀 repartition on high-cardinality composite key
demand_data = inv.repartition("part_num", "color_id")

# How bad is it?
num_parts = demand_data.rdd.getNumPartitions()
print(f"\U0001f4a5 Partition count after repartition(part_num, color_id): {num_parts:,}")

# Show partition size distribution — most will be empty or tiny
partition_dist = (
    demand_data
    .withColumn("_pid", spark_partition_id())
    .groupBy("_pid").count()
)

non_empty = partition_dist.count()
empty_count = num_parts - non_empty
print(f"   Empty partitions: {empty_count:,}")
print(f"   Non-empty partitions: {non_empty:,}")

# Baseline timing — scheduler overhead dominates
demand_forecast = demand_data.groupBy("part_num", "color_id").agg(
    F.sum("quantity").alias("TotalQuantity"),
    F.count("*").alias("TxnCount"),
)

demand_forecast.benchmark("INC-4: Demand Forecast", "before (exploded)").count()

In [ ]:
# ============================================================
# INC-4: THE FIX — Use repartition with a reasonable target count
# ============================================================

inv = spark.table(f"{CAPSTONE_SCHEMA}.inventory_transaction")

# ✅ Fixed: repartition with a reasonable target count
demand_data_fixed = inv.repartition(200)

num_parts_fixed = demand_data_fixed.rdd.getNumPartitions()
print(f"\u2705 Partition count after repartition(200): {num_parts_fixed:,}")

# Same aggregation, much less scheduling overhead
demand_forecast_fixed = demand_data_fixed.groupBy("part_num", "color_id").agg(
    F.sum("quantity").alias("TotalQuantity"),
    F.count("*").alias("TxnCount"),
)

demand_forecast_fixed.benchmark("INC-4: Demand Forecast", "after (repartition 200)").count()

---

## 🟢 INC-5: Delta Storage Regression — "Overnight Performance Drop" *(Optional)*

### 📋 On-Call Ticket

| Field | Details |
|-------|---------|
| **Severity** | 🟢 Low — pipeline slower but completing |
| **SLA Impact** | Manufacturing analytics dashboard loading 3× slower |
| **Symptom** | Query on `manufacturing_event` became 3× slower after yesterday's deployment |
| **Suspected Pipeline** | Manufacturing analytics aggregation |
| **Context** | A deployment yesterday changed table properties. No code changes to the query itself. |

### 🔍 Your Task

1. Run the query and capture baseline timing
2. Use `DESCRIBE HISTORY` to find what changed
3. Use `DESCRIBE DETAIL` to check file count and sizes
4. Check table properties — what was disabled?
5. Fix and validate

In [ ]:
# ============================================================
# INC-5: THE BROKEN QUERY — same code, slower after deployment
# ============================================================

me = spark.table(f"{CAPSTONE_SCHEMA}.manufacturing_event")

# Simple analytics query — aggregation by machine and defect type
mfg_analytics = (
    me
    .filter(F.col("defect_detected") == True)
    .groupBy("machine_id", "defect_type")
    .agg(
        F.count("*").alias("defect_count"),
        F.avg("mold_temp").alias("AvgMoldTemp"),
        F.avg("injection_pressure").alias("AvgPressure"),
        F.avg("cycle_time_ms").alias("AvgCycleTime"),
    )
    .orderBy(F.desc("defect_count"))
)

# Baseline timing
mfg_analytics.benchmark("INC-5: Mfg Analytics", "before (degraded)").count()

### 🎯 Challenge: What Changed?

The manufacturing analytics query was fast last week. Now it's slow. The code hasn't changed.

**Your task:** Investigate what happened:
1. Check `DESCRIBE HISTORY` for recent operations
2. Check `SHOW TBLPROPERTIES` for configuration changes
3. Check file metrics with `DESCRIBE DETAIL`

> 💡 Hint: Someone may have changed the table's optimization settings...

In [ ]:
# YOUR CODE HERE
# Investigate the manufacturing_event table
# spark.sql(f"DESCRIBE HISTORY {CAPSTONE_SCHEMA}.manufacturing_event").show(...)
# spark.sql(f"SHOW TBLPROPERTIES {CAPSTONE_SCHEMA}.manufacturing_event").show(...)

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
# Check history — look for SET TBLPROPERTIES operations
spark.sql(f"DESCRIBE HISTORY {CAPSTONE_SCHEMA}.manufacturing_event").select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

# Check current properties — what's disabled?
spark.sql(f"SHOW TBLPROPERTIES {CAPSTONE_SCHEMA}.manufacturing_event").show(20, truncate=False)
```

**Root cause:** Someone disabled both `autoCompact` and `optimizeWrite`. New data is landing as tiny files, causing the same small-file problem from Module 2!

</details>

---

In [ ]:
# ============================================================
# INC-5: DIAGNOSIS — What changed?
# ============================================================

print("=" * 60)
print("EVIDENCE 1: Table history — what happened recently?")
print("=" * 60)
spark.sql(f"DESCRIBE HISTORY {CAPSTONE_SCHEMA}.manufacturing_event").select(
    "version", "timestamp", "operation", "operationParameters"
).show(10, truncate=False)

print()
print("=" * 60)
print("EVIDENCE 2: Table properties — what's disabled?")
print("=" * 60)
spark.sql(f"SHOW TBLPROPERTIES {CAPSTONE_SCHEMA}.manufacturing_event").show(20, truncate=False)

print()
print("=" * 60)
print("EVIDENCE 3: File metrics")
print("=" * 60)
show_metrics(CAPSTONE_SCHEMA, "manufacturing_event", "current state")

print()
print("=" * 60)
print("EVIDENCE 4: How many files is the query scanning?")
print("=" * 60)
file_count = me.selectExpr("input_file_name()").distinct().count()
print(f"   Scanning {file_count:,} files")

In [ ]:
# ============================================================
# INC-5: THE FIX — Restore table properties and OPTIMIZE
# ============================================================

# Fix 1: Restore auto-optimization properties
print("\U0001f527 Fix 1: Restore table properties...")
spark.sql(f"""
    ALTER TABLE {CAPSTONE_SCHEMA}.manufacturing_event
    SET TBLPROPERTIES (
        'delta.autoOptimize.autoCompact' = 'true',
        'delta.autoOptimize.optimizeWrite' = 'true'
    )
""")
print("   Auto-compact and optimize write re-enabled.")

# Fix 2: Compact existing small files
print("\n\U0001f527 Fix 2: OPTIMIZE (compact existing small files)...")
spark.sql(f"OPTIMIZE {CAPSTONE_SCHEMA}.manufacturing_event")
show_metrics(CAPSTONE_SCHEMA, "manufacturing_event", "after OPTIMIZE")

In [ ]:
# ============================================================
# INC-5: VALIDATE — Re-run and compare
# ============================================================

me_fixed = spark.table(f"{CAPSTONE_SCHEMA}.manufacturing_event")

mfg_analytics_fixed = (
    me_fixed
    .filter(F.col("defect_detected") == True)
    .groupBy("machine_id", "defect_type")
    .agg(
        F.count("*").alias("defect_count"),
        F.avg("mold_temp").alias("AvgMoldTemp"),
        F.avg("injection_pressure").alias("AvgPressure"),
        F.avg("cycle_time_ms").alias("AvgCycleTime"),
    )
    .orderBy(F.desc("defect_count"))
)

mfg_analytics_fixed.benchmark("INC-5: Mfg Analytics", "after (OPTIMIZE + properties)").count()

In [ ]:
# 🏭 The manufacturing analytics we just fixed — real insights!
print("📊 Defect Rates by Machine\n")
display(spark.sql(f"""
    SELECT machine_id,
           COUNT(*) AS total_events,
           SUM(CASE WHEN defect_detected THEN 1 ELSE 0 END) AS defects,
           ROUND(SUM(CASE WHEN defect_detected THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS defect_rate_pct,
           ROUND(AVG(mold_temp), 1) AS avg_mold_temp
    FROM {CAPSTONE_SCHEMA}.manufacturing_event
    GROUP BY machine_id
    ORDER BY defect_rate_pct DESC
    LIMIT 10
"""))
print("💡 Older machines (lower IDs) tend to have higher defect rates — just like the real factory!")

---

## 📊 Capstone Summary Dashboard

Run the cell below to see your before/after results across all incidents.

In [ ]:
# ============================================================
# SUMMARY — All incident results
# ============================================================

print()
print("=" * 62)
print("  \U0001f9f1  MODULE 4 \u2014 CAPSTONE RESULTS")
print("=" * 62)

for scenario, states in benchmarks.items():
    if len(states) < 2:
        continue
    baseline_key = next(iter(states))
    baseline_ms = states[baseline_key]
    W = 58
    print()
    print(f"  \u250c{'\u2500' * W}\u2510")
    title = f"\033[1m{scenario}\033[0m"
    title_pad = W - 2 - len(scenario)
    print(f"  \u2502  {title}{' ' * title_pad}\u2502")
    print(f"  \u251c{'\u2500' * W}\u2524")
    print(f"  \u2502  {'State':<28}{'Time (ms)':>12}{'Factor':>14}  \u2502")
    print(f"  \u251c{'\u2500' * W}\u2524")
    for s, ms in states.items():
        ratio = baseline_ms / max(ms, 0.001)
        if s == baseline_key:
            visible_tag = "baseline"
            tag = visible_tag
        elif ratio > 1:
            visible_tag = f"{ratio:.1f}x faster"
            tag = f"\033[1;32m{visible_tag}\033[0m"
        else:
            visible_tag = f"{1/max(ratio,0.001):.1f}x slower"
            tag = f"\033[1;31m{visible_tag}\033[0m"
        pad = 14 - len(visible_tag)
        print(f"  \u2502  {s:<28}{ms:>12.2f}{' ' * pad}{tag}  \u2502")
    print(f"  \u2514{'\u2500' * W}\u2518")

print()
print("=" * 62)

---

## 🎯 Debrief — Key Takeaways

### The Production Triage Loop

Every incident you solved followed the same pattern — the skills from Modules 1–3
applied to real-world failure scenarios:

| Incident | Bottleneck Category | Module Connection |
|----------|-------------------|-------------------|
| INC-1: OOM Crash | **Memory** — broadcast too-large table | Module 1: Reading physical plans |
| INC-2: 10× Slowdown | **I/O + Statistics** — small files & stale stats | Module 2: OPTIMIZE + ANALYZE |
| INC-3: Mysterious Spill | **Memory + Skew** — uneven partition sizes | Module 0: Distribution & partitioning |
| INC-4: Partition Explosion | **Scheduling** — too many tiny tasks | Module 0: Parallelism tradeoffs |
| INC-5: Storage Regression | **Storage** — disabled optimizations & compaction | Module 2: Table properties & maintenance |

### Three Pillars of Spark Performance

All performance issues are variations of three fundamental problems:

1. **I/O Amplification** — Reading more data than necessary (small files, missing stats, no pruning)
2. **Memory Pressure** — Processing more data per task than memory allows (skew, bad broadcasts, unbounded state)
3. **Scheduling Overhead** — Coordination cost exceeds computation cost (too many partitions, tiny tasks)

### The Diagnostic Toolkit

| Tool | What It Tells You |
|------|-------------------|
| `explain()` / `explain(True)` | Join strategy, scan type, pushed filters |
| `DESCRIBE DETAIL` | File count, size, partitioning, table features |
| `DESCRIBE HISTORY` | What operations happened and when |
| `inputFiles()` | Which files a query will actually read |
| `getNumPartitions()` | How many partitions a DataFrame has |
| `spark_partition_id()` | Which rows are in which partition (skew detection) |
| `ANALYZE TABLE ... COMPUTE STATISTICS` | Refresh optimizer statistics |

---

**🎉 Congratulations!** You've completed the Advanced Spark Performance Engineering Workshop.

You can now diagnose, tune, and fix the most common Spark performance issues in production.